In [1]:
import mlflow
from mlflow.models import infer_signature
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
mlflow.set_tracking_uri(uri='http://localhost:5000')

In [3]:
X, y = datasets.load_iris(return_X_y=True)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
params = {
    'penalty': 'l2',  
    'dual': False,  
    'tol': 0.0001,  
    'C': 1.0,  
    'fit_intercept': True,  
    'intercept_scaling': 1,  
    'class_weight': None,  
    'random_state': None,  
    'solver': 'lbfgs',  
    'max_iter': 100,  
    'multi_class': 'deprecated',  
    'verbose': 1,  
    'warm_start': False,  
    'n_jobs': None,  
    'l1_ratio': None 
}

In [6]:
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)    

LogisticRegression(verbose=1)

In [7]:
y_pred = lr.predict(X_test)
y_pred

array([1, 0, 2, 1, 1, 0, 1, 2, 1, 1, 2, 0, 0, 0, 0, 1, 2, 1, 1, 2, 0, 2,
       0, 2, 2, 2, 2, 2, 0, 0])

In [8]:
accuracy = accuracy_score(y_test,y_pred)
print(f'Accuracy: {accuracy}')

Accuracy: 1.0


In [9]:
mlflow.set_experiment('Logistic Regression on iris dataset : ')

with mlflow.start_run():
    mlflow.log_params(params)
    mlflow.log_metric('accuracy', accuracy)
    mlflow.set_tag('model', 'Logistic Regression')

    signature = infer_signature(X_train, lr.predict(X_train) )

    model_info = mlflow.sklearn.log_model(
                                        sk_model=lr, 
                                        signature=signature,
                                        input_example=X_train,
                                        registered_model_name='logistic_model',
                                        artifact_path='logistic_model')

Registered model 'logistic_model' already exists. Creating a new version of this model...
2025/02/16 21:04:04 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: logistic_model, version 3


🏃 View run skittish-ape-370 at: http://localhost:5000/#/experiments/262651086410039652/runs/7f7149574c854986996aebb74b72f9fd
🧪 View experiment at: http://localhost:5000/#/experiments/262651086410039652


Created version '3' of model 'logistic_model'.


In [10]:
model_info.model_uri

'runs:/7f7149574c854986996aebb74b72f9fd/logistic_model'

In [11]:
import mlflow

model_uri = 'runs:/85a532bd131a4d3ab5f234feb1597df9/logistic_model'
# This is the input example logged with the model
pyfunc_model = mlflow.pyfunc.load_model(model_uri)
input_data = pyfunc_model.input_example

# Verify the model with the provided input data using the logged dependencies.
# For more details, refer to:
# https://mlflow.org/docs/latest/models.html#validate-models-before-deployment
mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_data,
    env_manager="uv",
)

2025/02/16 21:04:05 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'
2025/02/16 21:04:05 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/tmp5sk93zlw/envs/virtualenv_envs/mlflow-5ee0a220dc376a4ad0e7e96ea97c2310f6a3413b with python version 3.10.16 using uv
Using CPython 3.10.16 interpreter at: /home/ammar/anaconda3/envs/mlops/bin/python3.10
Creating virtual environment at: /tmp/tmp5sk93zlw/envs/virtualenv_envs/mlflow-5ee0a220dc376a4ad0e7e96ea97c2310f6a3413b
Activate with: source /tmp/tmp5sk93zlw/envs/virtualenv_envs/mlflow-5ee0a220dc376a4ad0e7e96ea97c2310f6a3413b/bin/activate
2025/02/16 21:04:05 INFO mlflow.utils.virtualenv: Installing dependencies
Using Python 3.10.16 environment at: /tmp/tmp5sk93zlw/envs/virtualenv_envs/mlflow-5ee0a220dc376a4ad0e7e96ea97c2310f6a3413b


KeyboardInterrupt: 

In [16]:
import mlflow
logged_model = 'runs:/85a532bd131a4d3ab5f234feb1597df9/logistic_model'

# Load model as a PyFuncModel.
loaded_model = mlflow.pyfunc.load_model(logged_model)
input_data = pyfunc_model.input_example

# Predict on a Pandas DataFrame.
loaded_model.predict(input_data)

array([0, 0, 1, 0, 0, 2, 1, 0, 0, 0, 2, 1, 1, 0, 0, 1, 2, 2, 1, 2, 1, 2,
       1, 0, 2, 1, 0, 0, 0, 1, 2, 0, 0, 0, 1, 0, 1, 2, 0, 1, 2, 0, 2, 2,
       1, 1, 2, 1, 0, 1, 2, 0, 0, 1, 2, 0, 2, 0, 0, 2, 1, 2, 2, 2, 2, 1,
       0, 0, 2, 2, 0, 0, 0, 1, 2, 0, 2, 2, 0, 1, 1, 2, 1, 2, 0, 2, 1, 2,
       1, 1, 1, 0, 1, 1, 0, 1, 2, 2, 0, 1, 2, 2, 0, 2, 0, 1, 2, 2, 1, 2,
       1, 1, 2, 2, 0, 1, 2, 0, 1, 2])